In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Sequence, Dict, Any

import numpy as np
import pandas as pd
from scipy.stats import norm

In [21]:
# ============================================================
# Input parameters
# ============================================================

@dataclass
class AssetPlan:
    """
    Parameters
    ----------
    p0 : float
        Initial asset amount P_0
    mu : float
        Mean of monthly log gross return, i.e. X_i ~ LN(mu, sigma^2)
    sigma : float
        Std dev of monthly log gross return
    n_months : int
        Investment horizon in months
    target_amount : float
        Goal amount T evaluated at the terminal month n
    cashflows : array-like of shape (n_months,), optional
        External cashflows c_i for i=1,...,n_months
        Positive = contribution, negative = withdrawal
    """
    p0: float
    mu: float
    sigma: float
    n_months: int
    target_amount: float
    cashflows: Optional[Sequence[float]] = None

    def __post_init__(self) -> None:
        if self.p0 < 0:
            raise ValueError("p0 must be non-negative.")
        if self.sigma < 0:
            raise ValueError("sigma must be non-negative.")
        if self.n_months <= 0:
            raise ValueError("n_months must be positive.")
        if self.target_amount < 0:
            raise ValueError("target_amount must be positive.")

        if self.cashflows is None:
            self.cashflows = np.zeros(self.n_months, dtype=float)
        else:
            self.cashflows = np.asarray(self.cashflows, dtype=float)
            if len(self.cashflows) != self.n_months:
                raise ValueError("cashflows must have length n_months.")

In [4]:
# ============================================================
# Utility functions
# ============================================================

def cumulative_withdrawal_floor(cashflows: np.ndarray) -> np.ndarray:
    """
    theta_i = cumulative withdrawals up to month i (<= 0)
    Positive cashflows are ignored, negative cashflows are accumulated.
    """
    negative_part = np.minimum(cashflows, 0.0)
    theta = np.zeros(len(cashflows) + 1, dtype=float)
    theta[1:] = np.cumsum(negative_part)
    return theta

In [5]:
def exceedance_prob_lognormal(target: float, log_mean: float, log_std: float) -> float:
    """
    Return P(Y > target) for Y ~ LogNormal(log_mean, log_std^2).
    Handles deterministic case log_std == 0.
    """
    if target <= 0:
        return 1.0

    if log_std <= 0:
        y = float(np.exp(log_mean))
        return 1.0 if y > target else 0.0

    z = (np.log(target) - log_mean) / log_std
    return float(1.0 - norm.cdf(z))

In [6]:
def exceedance_prob_shifted_lognormal(
    target: float,
    mu_hat: float,
    sigma_hat: float,
    theta: float,
) -> float:
    """
    Return P(P > target) for P ~ SLN(mu_hat, sigma_hat^2, -theta),
    i.e. log(P - theta) ~ N(mu_hat, sigma_hat^2).
    """
    if target <= theta:
        return 1.0

    if sigma_hat <= 0:
        p = float(np.exp(mu_hat) + theta)
        return 1.0 if p > target else 0.0

    z = (np.log(target - theta) - mu_hat) / sigma_hat
    return float(1.0 - norm.cdf(z))

In [7]:
# ============================================================
# 1) SLN approximation
# ============================================================

def sln_moment_recursion(plan: AssetPlan) -> Dict[str, np.ndarray]:
    """
    Equations corresponding to:
      P_i = P_{i-1} X_{i-1} + c_i
      E_i = exp(mu + sigma^2/2) * E_{i-1} + c_i
      V_i = exp(2(mu + sigma^2)) * V_{i-1}
            + exp(2mu + sigma^2) * (exp(sigma^2) - 1) * E_{i-1}^2

    Then infer SLN parameters from:
      sigma_hat_i^2 = log( V_i / (E_i - theta_i)^2 + 1 )
      mu_hat_i = log(E_i - theta_i) - sigma_hat_i^2 / 2
    """
    n = plan.n_months
    c = np.asarray(plan.cashflows, dtype=float)

    months = np.arange(n + 1)
    theta = cumulative_withdrawal_floor(c)

    E = np.zeros(n + 1, dtype=float)
    V = np.zeros(n + 1, dtype=float)

    E[0] = plan.p0
    V[0] = 0.0

    ex = np.exp(plan.mu + 0.5 * plan.sigma**2)
    vx = np.exp(2.0 * plan.mu + plan.sigma**2) * (np.exp(plan.sigma**2) - 1.0)
    ex2 = np.exp(2.0 * (plan.mu + plan.sigma**2))

    for i in range(1, n + 1):
        E[i] = ex * E[i - 1] + c[i - 1]
        V[i] = ex2 * V[i - 1] + vx * (E[i - 1] ** 2)
        V[i] = max(V[i], 0.0)

    mu_hat = np.full(n + 1, np.nan, dtype=float)
    sigma2_hat = np.full(n + 1, np.nan, dtype=float)

    for i in range(n + 1):
        shifted_mean = E[i] - theta[i]
        if shifted_mean <= 0:
            raise ValueError(
                f"At month {i}, E_i - theta_i <= 0, so SLN parameters are undefined."
            )
        sigma2_hat[i] = np.log(1.0 + V[i] / (shifted_mean ** 2))
        mu_hat[i] = np.log(shifted_mean) - 0.5 * sigma2_hat[i]

    sigma_hat = np.sqrt(np.maximum(sigma2_hat, 0.0))

    return {
        "months": months,
        "theta": theta,
        "mean": E,
        "variance": V,
        "std": np.sqrt(V),
        "mu_hat": mu_hat,
        "sigma2_hat": sigma2_hat,
        "sigma_hat": sigma_hat,
    }

In [8]:
def sln_terminal_goal_probability(plan: AssetPlan, sln_result: Dict[str, np.ndarray]) -> float:
    n = plan.n_months
    return exceedance_prob_shifted_lognormal(
        target=plan.target_amount,
        mu_hat=float(sln_result["mu_hat"][n]),
        sigma_hat=float(sln_result["sigma_hat"][n]),
        theta=float(sln_result["theta"][n]),
    )

In [9]:
# ============================================================
# 2) Monte Carlo simulation
# ============================================================

def monte_carlo_paths(
    plan: AssetPlan,
    n_sims: int = 100_000,
    seed: Optional[int] = 42,
) -> np.ndarray:
    """
    Simulate:
        P_i = P_{i-1} X_{i-1} + c_i
    with X_i ~ LN(mu, sigma^2), independent.
    """
    if n_sims <= 0:
        raise ValueError("n_sims must be positive.")

    rng = np.random.default_rng(seed)
    n = plan.n_months
    c = np.asarray(plan.cashflows, dtype=float)

    paths = np.empty((n_sims, n + 1), dtype=float)
    paths[:, 0] = plan.p0

    shocks = rng.normal(loc=plan.mu, scale=plan.sigma, size=(n_sims, n))
    gross_returns = np.exp(shocks)

    for i in range(1, n + 1):
        paths[:, i] = paths[:, i - 1] * gross_returns[:, i - 1] + c[i - 1]

    return paths

In [10]:
def monte_carlo_terminal_summary(plan: AssetPlan, paths: np.ndarray) -> Dict[str, float]:
    terminal = paths[:, -1]
    return {
        "mean": float(np.mean(terminal)),
        "variance": float(np.var(terminal, ddof=0)),
        "std": float(np.std(terminal, ddof=0)),
        "goal_probability": float(np.mean(terminal > plan.target_amount)),
    }

In [11]:
# ============================================================
# 3) Exact solution (only when all cashflows are zero)
# ============================================================

def exact_terminal_summary_no_cashflow(plan: AssetPlan) -> Optional[Dict[str, float]]:
    """
    If c_i = 0 for all i:
        P_n = P_0 * product_{j=0}^{n-1} X_j
    so:
        log P_n ~ N(log P_0 + n*mu, n*sigma^2)
    """
    c = np.asarray(plan.cashflows, dtype=float)
    if not np.allclose(c, 0.0):
        return None
    if plan.p0 <= 0:
        raise ValueError("Exact solution requires p0 > 0.")

    n = plan.n_months
    log_mean = np.log(plan.p0) + n * plan.mu
    log_var = n * (plan.sigma ** 2)
    log_std = np.sqrt(log_var)

    mean = float(np.exp(log_mean + 0.5 * log_var))
    variance = float((np.exp(log_var) - 1.0) * np.exp(2.0 * log_mean + log_var))
    std = float(np.sqrt(variance))
    goal_probability = exceedance_prob_lognormal(
        target=plan.target_amount,
        log_mean=log_mean,
        log_std=log_std,
    )

    return {
        "mean": mean,
        "variance": variance,
        "std": std,
        "goal_probability": float(goal_probability),
    }

In [12]:
# ============================================================
# Comparison
# ============================================================

def compare_methods(
    plan: AssetPlan,
    n_sims: int = 100_000,
    seed: Optional[int] = 42,
) -> Dict[str, Any]:
    """
    Compare:
      1) SLN approximation
      2) Monte Carlo
      3) Exact solution (only if all cashflows are zero)

    Returns
    -------
    dict with:
      - goal_probability_comparison : DataFrame
      - terminal_summary_comparison : DataFrame
      - raw results
    """
    sln = sln_moment_recursion(plan)
    sln_goal_prob = sln_terminal_goal_probability(plan, sln)

    mc_paths = monte_carlo_paths(plan, n_sims=n_sims, seed=seed)
    mc_terminal = monte_carlo_terminal_summary(plan, mc_paths)

    exact_terminal = exact_terminal_summary_no_cashflow(plan)

    terminal_rows = [
        {
            "method": "SLN",
            "terminal_mean": float(sln["mean"][-1]),
            "terminal_std": float(sln["std"][-1]),
            "goal_probability": float(sln_goal_prob),
        },
        {
            "method": "Monte Carlo",
            "terminal_mean": mc_terminal["mean"],
            "terminal_std": mc_terminal["std"],
            "goal_probability": mc_terminal["goal_probability"],
        },
    ]

    if exact_terminal is not None:
        terminal_rows.append(
            {
                "method": "Exact",
                "terminal_mean": exact_terminal["mean"],
                "terminal_std": exact_terminal["std"],
                "goal_probability": exact_terminal["goal_probability"],
            }
        )
    else:
        terminal_rows.append(
            {
                "method": "Exact",
                "terminal_mean": np.nan,
                "terminal_std": np.nan,
                "goal_probability": np.nan,
            }
        )

    terminal_summary_df = pd.DataFrame(terminal_rows)
    goal_probability_df = terminal_summary_df[["method", "goal_probability"]].copy()

    return {
        "plan": plan,
        "sln": sln,
        "mc_paths": mc_paths,
        "mc_terminal": mc_terminal,
        "exact_terminal": exact_terminal,
        "goal_probability_comparison": goal_probability_df,
        "terminal_summary_comparison": terminal_summary_df,
    }

In [21]:
# ============================================================
# Example usage
# ============================================================

def demo_no_cashflow() -> Dict[str, Any]:
    """
    All three methods are available.
    """
    plan = AssetPlan(
        p0=10_000_000,
        # mu=0.004951397394083888,
        # sigma=0.02435169546589359,
        mu=0.065/12,
        sigma=0.09/(12**0.5),
        n_months=120,
        target_amount=20_000_000,
        cashflows=np.zeros(120),
    )

    results = compare_methods(
        plan=plan,
        n_sims=200_000,
        seed=123,
    )

    print("=== Goal Probability Comparison ===")
    print(results["goal_probability_comparison"])
    print()

    print("=== Terminal Summary Comparison ===")
    print(results["terminal_summary_comparison"])
    print()

    return results

In [13]:
def demo_with_cashflows() -> Dict[str, Any]:
    """
    Exact solution is unavailable because cashflows are non-zero.
    """
    n_months = 120
    n_plus = 60
    c_plus = 50_000.0
    c_minus = -80_000.0

    cashflows = np.array(
        [c_plus] * n_plus + [c_minus] * (n_months - n_plus),
        dtype=float,
    )

    plan = AssetPlan(
        p0=10_000_000,
        mu=0.0030,
        sigma=0.0450,
        n_months=n_months,
        target_amount=12_000_000,
        cashflows=cashflows,
    )

    results = compare_methods(
        plan=plan,
        n_sims=200_000,
        seed=123,
    )

    print("=== Goal Probability Comparison ===")
    print(results["goal_probability_comparison"])
    print()

    print("=== Terminal Summary Comparison ===")
    print(results["terminal_summary_comparison"])
    print()

    return results

In [14]:
# ============================================================
# リターン6.5%、リスク9%、連続複利変換なし、120ヶ月、目標200万円、初期資産100万円、積立なし、取崩なし
# ============================================================
plan = AssetPlan(
    p0=1_000_000,
    mu=0.065/12,
    sigma=0.09/(12**0.5),
    n_months=120,
    target_amount=2_000_000,
    cashflows=np.zeros(120),
)
results = compare_methods(
    plan=plan,
    n_sims=200_000,
    seed=123,
)
print("=== Goal Probability Comparison ===")
print(results["goal_probability_comparison"])


=== Goal Probability Comparison ===
        method  goal_probability
0          SLN          0.439750
1  Monte Carlo          0.440315
2        Exact          0.439750


In [18]:
# ============================================================
# リターン6.5%、リスク9%、連続複利変換あり、120ヶ月、目標200万円、初期資産100万円、積立なし、取崩なし
# ============================================================
plan = AssetPlan(
    p0=1_000_000,
    mu=0.004951397394083888,
    sigma=0.02435169546589359,
    n_months=120,
    target_amount=2_000_000,
    cashflows=np.zeros(120),
)
results = compare_methods(
    plan=plan,
    n_sims=200_000,
    seed=123,
)
print("=== Goal Probability Comparison ===")
print(results["goal_probability_comparison"])

=== Goal Probability Comparison ===
        method  goal_probability
0          SLN          0.355302
1  Monte Carlo          0.356190
2        Exact          0.355302


In [17]:
# ============================================================
# リターン6.5%、リスク9%、連続複利変換なし、120ヶ月、目標300万円、初期資産100万円、積立あり、取崩なし
# ============================================================
n_months=120
monthly_contribution = 10_000
cashflows = np.zeros(n_months)
# 2026/04 ～ 2036/02 = c1 ～ c119
cashflows[:119] = monthly_contribution
plan = AssetPlan(
    p0=1_000_000,
    mu=0.065/12,
    sigma=0.09/(12**0.5),
    n_months=120,
    target_amount=3_000_000,
    cashflows=cashflows,
)
results = compare_methods(
    plan=plan,
    n_sims=200_000,
    seed=123,
)
print("=== Goal Probability Comparison ===")
print(results["goal_probability_comparison"])

=== Goal Probability Comparison ===
        method  goal_probability
0          SLN           0.78985
1  Monte Carlo           0.79214
2        Exact               NaN


In [19]:
# ============================================================
# リターン6.5%、リスク9%、連続複利変換あり、120ヶ月、目標300万円、初期資産100万円、積立あり、取崩なし
# ============================================================
n_months=120
monthly_contribution = 10_000
cashflows = np.zeros(n_months)
# 2026/04 ～ 2036/02 = c1 ～ c119
cashflows[:119] = monthly_contribution
plan = AssetPlan(
    p0=1_000_000,
    mu=0.004951397394083888,
    sigma=0.02435169546589359,
    n_months=120,
    target_amount=3_000_000,
    cashflows=cashflows,
)
results = compare_methods(
    plan=plan,
    n_sims=200_000,
    seed=123,
)
print("=== Goal Probability Comparison ===")
print(results["goal_probability_comparison"])

=== Goal Probability Comparison ===
        method  goal_probability
0          SLN          0.744471
1  Monte Carlo          0.745220
2        Exact               NaN


In [23]:
# ============================================================
# リターン6.5%、リスク9%、連続複利変換なし、120ヶ月、目標0万円、初期資産200万円、積立なし、取崩あり
# ============================================================
n_months=121
monthly_withdrawal = 20_000
cashflows = np.zeros(n_months)
cashflows[:120] = -monthly_withdrawal
plan = AssetPlan(
    p0=2_000_000,
    mu=0.065/12,
    sigma=0.09/(12**0.5),
    n_months=121,
    target_amount=0,
    cashflows=cashflows,
)
results = compare_methods(
    plan=plan,
    n_sims=200_000,
    seed=123,
)
print("=== Goal Probability Comparison ===")
print(results["goal_probability_comparison"])

=== Goal Probability Comparison ===
        method  goal_probability
0          SLN          0.783084
1  Monte Carlo          0.789265
2        Exact               NaN


In [24]:
# ============================================================
# リターン6.5%、リスク9%、連続複利変換あり、120ヶ月、目標0万円、初期資産200万円、積立なし、取崩あり
# ============================================================
n_months=121
monthly_withdrawal = 20_000
cashflows = np.zeros(n_months)
cashflows[:120] = -monthly_withdrawal
plan = AssetPlan(
    p0=2_000_000,
    mu=0.004951397394083888,
    sigma=0.02435169546589359,
    n_months=121,
    target_amount=0,
    cashflows=cashflows,
)
results = compare_methods(
    plan=plan,
    n_sims=200_000,
    seed=123,
)
print("=== Goal Probability Comparison ===")
print(results["goal_probability_comparison"])

=== Goal Probability Comparison ===
        method  goal_probability
0          SLN          0.750400
1  Monte Carlo          0.752555
2        Exact               NaN
